<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-07/AI_Modul_7.3_Praktikum_K_Nearest_Neighbor.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 7.3: Praktikum K-Nearest Neighbor (KNN)

**Mata Kuliah:** Kecerdasan Buatan & Pembelajaran Mesin Terapan  
**Institusi:** Institut Pertanian Stiper (INSTIPER) Yogyakarta  
**Studi Kasus:** Pemetaan Spasial Serangan Hama Ulat Api (*Setothosea asigna*) pada Perkebunan Kelapa Sawit Berbasis Data Multispektral Drone

---

### Capaian Pembelajaran Praktikum
1. Mampu mengimplementasikan algoritma K-Nearest Neighbor secara manual menggunakan komputasi matriks jarak NumPy.
2. Mampu membangun pipeline klasifikasi KNN menggunakan Scikit-Learn lengkap dengan `StandardScaler`.
3. Mampu melakukan pencarian hiperparameter nilai $K$ optimal dan komparasi skema pembobotan (`uniform` vs `distance`).
4. Mampu mendemonstrasikan secara empiris dampak buruk fitur bising dan fenomena *Curse of Dimensionality* terhadap kinerja KNN.

## 1. Import Pustaka & Konfigurasi Grafis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Pengaturan visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 120

print("[OK] Seluruh modul analisis data berhasil dimuat!")

## 2. Pembangkitan Dataset Penginderaan Jauh Hama Sawit

Fitur kanopi tanaman sawit:
- `Reflektansi_NIR`: Pantulan cahaya inframerah dekat (skala 0 - 100)
- `Indeks_NDVI`: Normalized Difference Vegetation Index (skala 0.0 - 1.0)
- `Kerapatan_Tajuk`: Persentase ketebalan tajuk daun fotosintetik (%)
- `Status_Hama`: Label biner (1 = Terserang Ulat Api, 0 = Sehat)

In [ ]:
np.random.seed(42)
n_samples = 260

# 1. Pohon Sehat (Kelas 0, 60% populasi)
n0 = int(n_samples * 0.60)
nir_0 = np.random.normal(74.0, 5.5, n0)
ndvi_0 = np.random.normal(0.79, 0.04, n0)
tajuk_0 = np.random.normal(84.0, 4.5, n0)
y0 = np.zeros(n0, dtype=int)

# 2. Pohon Terserang Ulat Api (Kelas 1, 40% populasi - defoliasi kanopi)
n1 = n_samples - n0
nir_1 = np.random.normal(49.0, 7.5, n1)
ndvi_1 = np.random.normal(0.51, 0.07, n1)
tajuk_1 = np.random.normal(53.0, 8.5, n1)
y1 = np.ones(n1, dtype=int)

df_sawit = pd.DataFrame({
    'Reflektansi_NIR': np.round(np.concatenate([nir_0, nir_1]), 1),
    'Indeks_NDVI': np.round(np.concatenate([ndvi_0, ndvi_1]), 3),
    'Kerapatan_Tajuk': np.round(np.concatenate([tajuk_0, tajuk_1]), 1),
    'Status_Hama': np.concatenate([y0, y1])
})

print("Distribusi Kelas Sampel:")
print(df_sawit['Status_Hama'].value_counts())
display(df_sawit.head())

## 3. Eksplorasi Visual: Sebaran Data & Klaster Non-Linier

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    df_sawit['Reflektansi_NIR'],
    df_sawit['Indeks_NDVI'],
    c=df_sawit['Status_Hama'],
    cmap='coolwarm_r',
    s=df_sawit['Kerapatan_Tajuk'] * 0.8,
    edgecolors='k',
    alpha=0.75
)
ax.set_title("Sebaran Biofisik Kanopi Sawit (Ukuran Titik = Kerapatan Tajuk)", fontweight='bold')
ax.set_xlabel("Reflektansi NIR")
ax.set_ylabel("Indeks NDVI")

cbar = plt.colorbar(scatter, ticks=[0, 1])
cbar.set_ticklabels(['Sehat (0)', 'Terserang Hama (1)'])
plt.show()

## 4. Partisi Data & Standardisasi Fitur Mutlak (`StandardScaler`)

KNN sangat sensitif terhadap skala fitur. Tanpa standardisasi, fitur `Reflektansi_NIR` (skala ~70) akan mendominasi `Indeks_NDVI` (skala ~0.7) hingga seratus kali lipat!

In [ ]:
fitur = ['Reflektansi_NIR', 'Indeks_NDVI', 'Kerapatan_Tajuk']
target = 'Status_Hama'

X = df_sawit[fitur]
y = df_sawit[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Ukuran Latih  : {X_train_scaled.shape}")
print(f"Ukuran Uji    : {X_test_scaled.shape}")

## 5. Implementasi Algoritma KNN Manual dari Nol (NumPy Vektor)

Formula jarak Euclidean:
$$d(\mathbf{x}_q, \mathbf{x}_i) = \sqrt{\sum_{j=1}^p (x_{qj} - x_{ij})^2}$$

In [ ]:
class KNNScratch:
    def __init__(self, k: int = 5, metric: str = 'euclidean'):
        self.k = k
        self.metric = metric
        self.X_train = None
        self.y_train = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        # KNN adalah Lazy Learner: hanya menghafal data ke memori
        self.X_train = np.array(X)
        self.y_train = np.array(y)

    def _compute_distance(self, x_query: np.ndarray) -> np.ndarray:
        if self.metric == 'euclidean':
            return np.sqrt(np.sum((self.X_train - x_query)**2, axis=1))
        elif self.metric == 'manhattan':
            return np.sum(np.abs(self.X_train - x_query), axis=1)
        else:
            raise ValueError("Metrik tidak didukung")

    def predict_single(self, x_query: np.ndarray) -> int:
        # 1. Hitung jarak ke seluruh sampel latih
        distances = self._compute_distance(x_query)
        # 2. Ambil K indeks tetangga terdekat
        k_indices = np.argsort(distances)[:self.k]
        # 3. Ambil label k-tetangga
        k_nearest_labels = self.y_train[k_indices]
        # 4. Majority voting
        majority_label = np.bincount(k_nearest_labels).argmax()
        return majority_label

    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.array([self.predict_single(x) for x in X])

# Pengujian model scratch dengan K=5
knn_scratch = KNNScratch(k=5, metric='euclidean')
knn_scratch.fit(X_train_scaled, y_train.values)
pred_scratch = knn_scratch.predict(X_test_scaled)

print("[OK] Model KNN NumPy Scratch berhasil memprediksi data uji!")

## 6. Pemodelan Menggunakan Scikit-Learn & Verifikasi Ekuivalensi

In [ ]:
knn_sklearn = KNeighborsClassifier(n_neighbors=5, metric='euclidean', algorithm='brute')
knn_sklearn.fit(X_train_scaled, y_train)
pred_sklearn = knn_sklearn.predict(X_test_scaled)

# Perbandingan kecocokan prediksi manual vs Scikit-Learn
kecocokan = np.mean(pred_scratch == pred_sklearn) * 100
print(f"Tingkat Ekuivalensi NumPy Scratch vs Scikit-Learn: {kecocokan:.2f}%")
assert kecocokan == 100.0, "Hasil implementasi manual tidak identik!"
print("[VERIFIED] Hasil komputasi jarak dan voting manual 100% identik dengan Scikit-Learn!")

## 7. Pencarian Hiperparameter K Optimal (Elbow Method & Cross-Validation)

In [ ]:
k_values = range(1, 31, 2)  # Hanya menguji K ganjil untuk mencegah tie vote
cv_scores = []
error_rates = []

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for k in k_values:
    model_k = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    scores = cross_val_score(model_k, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    cv_scores.append(scores.mean())
    error_rates.append(1.0 - scores.mean())

best_k = k_values[np.argmax(cv_scores)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_values, error_rates, color='#d62728', marker='o', linewidth=2.5, label='Cross-Validation Error Rate')
ax.axvline(best_k, color='#2ca02c', linestyle='--', linewidth=2, label=f'K Optimal = {best_k} (Akurasi: {max(cv_scores)*100:.2f}%)')
ax.set_title("Evaluasi Nilai K Optimal Berbasis Validasi Silang (5-Fold)", fontweight='bold')
ax.set_xlabel("Nilai Hiperparameter K")
ax.set_ylabel("Rerata Tingkat Galat (Error Rate)")
ax.set_xticks(list(k_values))
ax.legend()
plt.show()

print(f"Nilai K Terbaik Terpilih : K = {best_k}")

## 8. Komparasi Skema Pembobotan: `uniform` vs `distance`

In [ ]:
model_uniform = KNeighborsClassifier(n_neighbors=best_k, weights='uniform').fit(X_train_scaled, y_train)
model_distance = KNeighborsClassifier(n_neighbors=best_k, weights='distance').fit(X_train_scaled, y_train)

acc_uniform = accuracy_score(y_test, model_uniform.predict(X_test_scaled))
acc_distance = accuracy_score(y_test, model_distance.predict(X_test_scaled))

print("=== KOMPARASI SKEMA PEMBOBOTAN PADA DATA UJI ===")
print(f"Akurasi Uniform Voting  (Setara)      : {acc_uniform*100:.2f}%")
print(f"Akurasi Distance-Weighted (Terbobot) : {acc_distance*100:.2f}%")

print("\nLaporan Klasifikasi Model Terbaik (Distance-Weighted):")
print(classification_report(y_test, model_distance.predict(X_test_scaled), target_names=['Sehat', 'Hama']))

## 9. Demonstrasi Empiris: Bahaya Kutukan Dimensi (*Curse of Dimensionality*)

Kita akan menyimulasikan penyuntikan 40 fitur acak murni yang tidak memiliki hubungan biologis dengan hama untuk membuktikan bagaimana fitur berlebih mendegradasi performa KNN.

In [ ]:
np.random.seed(42)
# Injeksi 40 fitur bising (noise) ke data latih dan uji
noise_train = np.random.normal(0, 1, size=(X_train_scaled.shape[0], 40))
noise_test = np.random.normal(0, 1, size=(X_test_scaled.shape[0], 40))

X_train_noisy = np.hstack([X_train_scaled, noise_train])
X_test_noisy = np.hstack([X_test_scaled, noise_test])

model_noisy = KNeighborsClassifier(n_neighbors=best_k, weights='distance')
model_noisy.fit(X_train_noisy, y_train)
acc_noisy = accuracy_score(y_test, model_noisy.predict(X_test_noisy))

print("=== SIMULASI KUTUKAN DIMENSI (CURSE OF DIMENSIONALITY) ===")
print(f"Akurasi KNN pada 3 Fitur Asli Drone        : {acc_distance*100:.2f}%")
print(f"Akurasi KNN setelah Ditambah 40 Fitur Bising : {acc_noisy*100:.2f}%")
print(f"Degradasi Akurasi Akibat Kutukan Dimensi     : {(acc_distance - acc_noisy)*100:.2f}% (Turun Drastis!)")

## 10. Tugas Mandiri & Eksplorasi Mahasiswa

1. Uji sebuah sampel pohon sawit baru dengan hasil drone: NIR = 55.0, NDVI = 0.58, Kerapatan Tajuk = 60.0%. Berapakah prediksi statusnya menggunakan model `model_distance`?
2. Bandingkan performa klasifikasi jika metrik jarak diubah dari Euclidean menjadi Manhattan (`metric='manhattan'`). Apakah terjadi perubahan performa?
3. Jelaskan mengapa pada algoritma KNN, penambahan fitur bising menyebabkan penurunan performa yang jauh lebih parah dibandingkan pada model Regresi Linier berpenalti!